# Local Mask R-CNN Training

현재 프로젝트의 `images/`와 `merged_annotations.json`을 직접 사용합니다. Google Drive 및 Colab 코드는 사용하지 않습니다. 실제 학습 구현은 `training/train_mask_rcnn.py`에 있으며 이 노트북은 환경 확인, 학습 실행, 결과 확인용입니다.

In [ ]:
from pathlib import Path
import json, torch, torchvision

PROJECT_ROOT = Path.cwd()
IMAGE_DIR = PROJECT_ROOT / "images"
ANNOTATION_FILE = PROJECT_ROOT / "merged_annotations.json"
OUTPUT_DIR = PROJECT_ROOT / "output" / "mask_rcnn"

print("project:", PROJECT_ROOT)
print("images:", IMAGE_DIR)
print("annotations:", ANNOTATION_FILE)
print("output:", OUTPUT_DIR)
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("cuda:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
assert IMAGE_DIR.is_dir()
assert ANNOTATION_FILE.is_file()
assert torch.cuda.is_available()

## 데이터 점검

In [ ]:
data = json.loads(ANNOTATION_FILE.read_text(encoding="utf-8"))
missing = [x["file_name"] for x in data["images"] if not (IMAGE_DIR / x["file_name"]).is_file()]
print("COCO images:", len(data["images"]))
print("annotations:", len(data["annotations"]))
print("categories:", data["categories"])
print("missing:", len(missing))
assert not missing, missing[:10]

## 학습 실행

RTX 4050 Laptop 6GB 기준 기본 설정은 batch 1, 640–1024 resize, AMP, 10 epochs입니다. 터미널에서 실행할 때도 동일한 명령을 사용합니다.

In [ ]:
%run training/train_mask_rcnn.py \
    --images images \
    --annotations merged_annotations.json \
    --output output/mask_rcnn \
    --epochs 10 \
    --batch-size 1 \
    --workers 0 \
    --min-size 640 \
    --max-size 1024

## 학습 결과

In [ ]:
print(list(OUTPUT_DIR.glob("*")))
metadata = json.loads((OUTPUT_DIR / "model_metadata.json").read_text(encoding="utf-8"))
metadata